# Radial Basis Function Network (RBFN) - TensorFlow / Keras

**Goal:** Approximate a nonlinear regression function.

This notebook favors clear, production-style structure: seeded runs,
explicit data preparation, small reusable modules, and compact
training loops that can be expanded for larger experiments.


## Architecture Notes

- **What it learns:** Gaussian basis functions convert distances to smooth nonlinear features.
- **Where it is used:** smooth function approximation and compact nonlinear regressors.
- **Why it works:** the architecture builds a useful bias into the computation, so the model does not need to rediscover that structure from data alone.
- **Output to expect:** classification models return class scores/probabilities, reconstruction models return reconstructed inputs, and generative models return new or denoised samples.


## Visual Intuition

Run this cell before or after training. It is lightweight and framework-independent, so it explains the network idea without requiring a long training run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
fig.suptitle("Radial Basis Function Network: intuition, signal flow, and output", fontsize=13)
axes[0].axis("off")
layers = ['x', 'RBFs', 'y']
xs = np.linspace(0.1, 0.9, len(layers))
for xpos, label in zip(xs, layers):
    axes[0].scatter([xpos], [0.55], s=1200, color="#4C78A8", alpha=0.18, edgecolors="#4C78A8")
    axes[0].text(xpos, 0.55, label, ha="center", va="center", fontsize=9)
for a, b in zip(xs[:-1], xs[1:]):
    axes[0].annotate("", xy=(b - 0.045, 0.55), xytext=(a + 0.045, 0.55), arrowprops=dict(arrowstyle="->", lw=1.5))
axes[0].set_title("How data moves")
x = np.linspace(-3, 3, 160)
y = np.exp(-(x-1)**2)-.7*np.exp(-(x+1)**2)
axes[1].plot(x, y, color="#F58518", lw=2)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("Toy behavior")
axes[1].grid(alpha=0.25)
values = np.array([.05,.02])
axes[2].bar(range(len(values)), values, color=["#54A24B", "#E45756", "#72B7B2", "#B279A2"][:len(values)])
axes[2].set_title("Typical output")
axes[2].set_xticks(range(len(values)))
axes[2].set_xticklabels(['train', 'test'])
axes[2].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)
print(f"TensorFlow: {tf.__version__}")
X = np.linspace(-3, 3, 240, dtype="float32")[:, None]
y = (np.sin(2 * X) + 0.2 * X).astype("float32")


In [ ]:
class RBFLayer(layers.Layer):
    def __init__(self, centers=24):
        super().__init__()
        self.centers_count = centers

    def build(self, input_shape):
        self.centers = self.add_weight(shape=(self.centers_count,), initializer="random_normal", trainable=True)
        self.log_gamma = self.add_weight(shape=(self.centers_count,), initializer="zeros", trainable=True)

    def call(self, inputs):
        distances = tf.square(inputs - self.centers[None, :])
        return tf.exp(-tf.exp(self.log_gamma)[None, :] * distances)


model = keras.Sequential([layers.Input(shape=(1,)), RBFLayer(24), layers.Dense(1)])
model.compile(optimizer=keras.optimizers.AdamW(3e-3), loss="mse")
model.fit(X, y, epochs=300, batch_size=32, verbose=0)
print("Final MSE:", model.evaluate(X, y, verbose=0))
